# Processing results for FHIR retrieval benchmark

Install the package from the repository root before running this notebook:

```bash
pip install -e .
```

This notebook reads run directories written by `scripts/run_evals.py` (each containing `config.yaml` and `results.json`).

In [ ]:
# @title Kernel Reproducibility
print("Local runtime")

In [ ]:
import os
import json
import random
import re
import jinja2
import tqdm
import asyncio
import pandas as pd
import numpy as np
import datetime
import requests
import uuid
from typing import Any, Generator, List, Optional, Tuple
from absl import logging
import pydantic
import pickle
import seaborn as sns
import time
import matplotlib.pyplot as plt
import yaml
from IPython.display import display, HTML

from fhir_retrieval_bench.data import fhir_utils

In [ ]:
# @title Matplotlib Styling
import matplotlib as mpl

SMALL_SIZE = 16
MEDIUM_SIZE = 18
BIGGER_SIZE = 20
mpl.rcParams['figure.dpi'] = 100

# Set general seaborn styles
sns.set_style(
    'white',
    {
        'font.family': ['sans-serif'],
    },
)
plt.rc('axes.spines', top=False, right=False)

# Load results

In [ ]:
def flatten_dict(data):
  flattened = {}
  for key, value in data.items():
    if isinstance(value, dict):
      flattened.update({f"{key}_{subkey}": subvalue for subkey, subvalue in flatten_dict(value).items()})
    else:
      flattened[key] = value
  return flattened

def load_run_dirs(run_dir_path):
  run_name = os.path.basename(run_dir_path)
  config_path = os.path.join(run_dir_path, "config.yaml")
  checkpoint_path = os.path.join(run_dir_path, "checkpoint.jsonl")
  results_path = os.path.join(run_dir_path, "results.json")

  try:
    with open(config_path, "r") as f:
      config_data = yaml.safe_load(f)
    if not os.path.exists(results_path):
      if os.path.exists(checkpoint_path):
        results_data = []
        with open(checkpoint_path, "r") as f:
          for line in f:
            if line.strip():
              results_data.append(json.loads(line))
      else:
        return None
    else:
      with open(results_path, "r") as f:
        results_data = json.load(f)

    config_flattened = flatten_dict(config_data)
    config_flattened["run_name"] = run_name
    config_flattened = {f"config_{key}": value for key, value in config_flattened.items()}
    for result in results_data:
      result.update(config_flattened)
    return results_data
  except Exception:
    return None

# Each base dir below holds run dirs directly at depth 1 (no nested results_* parents).
base_dirs = [
    'outputs/current_run/',
]

import glob
run_dirs = []
for b_dir in base_dirs:
    run_dirs += glob.glob(os.path.join(b_dir, '*'))

# Keep only directories that actually look like run dirs (have a config.yaml).
run_dirs = [
    d for d in set(run_dirs)
    if os.path.isdir(d) and os.path.exists(os.path.join(d, 'config.yaml'))
]
print(f"Found {len(run_dirs)} unique run directories.")

results = []
for rd in tqdm.tqdm(run_dirs, desc="Loading run results"):
  res = load_run_dirs(rd)
  if res:
    results.extend(res)

if results:
  results_df = pd.DataFrame(results)
  
  # Normalize the bench's `ontology_guided_retrieval` strategy name to the
  # canonical `semantic_rag` that figure code expects. NOTE: `flowsheet` and
  # `flowsheet_agent` are two distinct strategies and
  # must be kept distinct.
  results_df['config_strategy_name'] = results_df['config_strategy_name'].replace({
      'ontology_guided_retrieval': 'semantic_rag',
  })
  
  results_df = results_df.sort_values(["config_dataset_name", "instance_id", "config_run_name"])
  
  # Surface the hyperparams that vary across our runs so they're easy to filter on.
  hparam_cols = [
      'config_dataset_name',
      'config_strategy_name',
      'config_models_answer_model',
      'config_models_answer_temperature',
      'config_strategy_ns_agent_saliency_threshold',
      'config_strategy_recency_weight',
      'config_strategy_max_linearization_tokens',
      'config_strategy_embedding_csl_saliency_threshold',
      'config_run_name',
  ]
  present_hparam_cols = [c for c in hparam_cols if c in results_df.columns]
  print('\nUnique runs (one row per (dataset, strategy, model, hparams, run_name)):')
  display(
      results_df[present_hparam_cols]
      .drop_duplicates()
      .sort_values(present_hparam_cols)
      .reset_index(drop=True)
  )
  display(results_df.head())
else:
  print("No results loaded.")

In [ ]:
print("Unique strategies in results_df:", results_df['config_strategy_name'].unique())
print("Total rows for 'flowsheet':", len(results_df[results_df['config_strategy_name'] == 'flowsheet']))

# Summarize the results

In [ ]:
# @title Find common set of instances (filter out system-level failure runs)

def get_all_strategy_model_configs(rows):
  return set(rows.apply(lambda x: (x["config_strategy_name"], x["config_models_answer_model"]), axis=1).values)

# All (strategy, answer_model) configs
all_strategy_model_configs = get_all_strategy_model_configs(results_df)

# For each instance, see if results for all configs exist
common_instances_check = results_df[results_df["run_error"]!='NS Agent did not return a response']\
  .groupby(["config_dataset_name", "instance_id"])\
  .apply(
      lambda x: get_all_strategy_model_configs(x) == all_strategy_model_configs,
      include_groups=False
  )
common_instances = common_instances_check[common_instances_check].index


# convert to a dict that maps dataset names to list of available instances.
common_instances_per_dataset = {}

for dataset, instance in common_instances:
  if dataset not in common_instances_per_dataset:
    common_instances_per_dataset[dataset] = []
  common_instances_per_dataset[dataset].append(instance)

for dataset in common_instances_per_dataset:
  print(dataset, len(common_instances_per_dataset[dataset]))

In [ ]:
# @title Compute summary statistics

def bootstrap_values(data, statistic=np.mean, n_bootstrap=1000, ci=95, random_state=None):
    rng = np.random.default_rng(random_state)
    data = np.array(data)
    n = len(data)
    stats = []
    for _ in range(n_bootstrap):
        sample = rng.choice(data, size=n, replace=True)
        stats.append(statistic(sample))
    alpha = 100 - ci
    lower = np.percentile(stats, alpha / 2)
    upper = np.percentile(stats, 100 - alpha / 2)
    margin = (upper - lower) / 2
    return statistic(data), (lower, upper), margin

def calculate_summay_stats(group):
    group = group.copy()
    group["is_correct"] = pd.to_numeric(group["is_correct"], errors='coerce').fillna(0).astype(int)
    bs_mean, bs_ci, bs_margin = bootstrap_values(group["is_correct"], statistic=np.mean)
    return pd.Series({
        "accuracy": bs_mean,
        "accuracy_margin": bs_margin,
        "correct_count": group["is_correct"].sum(),
        "total_count": len(group),
        "mean_context_token_count": group["context_token_count"].mean(),
        "context_limit": group["context_window_limit"].mean(),
        "mean_context_usage_pct": group["context_fit_pct"].clip(lower=0.0, upper=1.0).mean(),
        "error_types": group["run_error"].value_counts().to_dict(),
        "error_types_pct": (group["run_error"].value_counts()/len(group)).to_dict(),
        "run_time (sec)": group["execution_time"].mean(),
    })

# Count 'Context too long' as wrong (non-retryable); drop other errors (retryable).
_err = results_df["run_error"].fillna("").astype(str).str.strip()
common_mask = results_df.apply(
    lambda x: common_instances_check[x["config_dataset_name"], x["instance_id"]], axis=1
).values
results_df_common = results_df[common_mask & ((_err.values == "") | (_err.values == "Context too long"))]

# Group by config_run_name + hparams so per-run rows are distinct.
summary_df_plot = results_df_common.groupby(
    [
        "config_dataset_name",
        "config_strategy_name",
        "config_models_answer_model",
        "config_models_answer_temperature",
        "config_strategy_ns_agent_saliency_threshold",
        "config_strategy_recency_weight",
        "config_strategy_max_linearization_tokens",
        "config_run_name",
    ],
    dropna=False,
).apply(calculate_summay_stats, include_groups=False)

In [ ]:
results_df_common

In [ ]:
summary_df_plot

In [ ]:
# @title Display formatted Summary Table (Common Instances)
summary_df_clean = summary_df_plot.reset_index()

# Sort by dataset, then by highest accuracy
summary_df_clean = summary_df_clean.sort_values(
    by=["config_dataset_name", "accuracy"],
    ascending=[True, False]
)

# Format the output for easier reading in Colab
styled_df = summary_df_clean.style.format({
    "accuracy": "{:.1%}",
    "accuracy_margin": "±{:.1%}",
    "mean_context_usage_pct": "{:.1%}",
    "run_time (sec)": "{:.2f}s"
}).background_gradient(subset=["accuracy"], cmap="Greens")

display(styled_df)

In [ ]:
# # @title Visualize Accuracy: Models vs Strategies
# import matplotlib.ticker as mtick

# df_plot = summary_df_common.reset_index()

# # Create a facet grid if you have multiple datasets
# g = sns.catplot(
#     data=df_plot,
#     x="config_strategy_name",
#     y="accuracy",
#     hue="config_models_answer_model",
#     col="config_dataset_name",
#     kind="bar",
#     height=6,
#     aspect=1.2,
#     palette="viridis"
# )

# # Add error bars manually using the bootstrapped margin
# for ax, dataset in zip(g.axes.flat, df_plot["config_dataset_name"].unique()):
#     ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
#     ax.set_title(f"Dataset: {dataset}")
#     ax.set_xlabel("Retrieval Strategy")
#     ax.set_ylabel("Accuracy")

#     # Optional: Add gridlines for readability
#     ax.grid(axis='y', linestyle='--', alpha=0.7)

# plt.subplots_adjust(top=0.9)
# g.figure.suptitle('FHIR Retrieval Benchmark: Accuracy by Strategy & Model', fontsize=16)
# plt.show()

# Generate paper figures

In [ ]:
PLOT_DIR = "outputs/paper_plots/"
os.makedirs(PLOT_DIR, exist_ok=True)

In [ ]:
# @title Best Model per Strategy (Modularized with Global Config)

import os
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.patches as mpatches
import seaborn as sns
from collections import OrderedDict

# ==============================================================================
# CONFIGURATION & MAPPINGS
# ==============================================================================

PRETTIFY_STRATEGY_NAME = OrderedDict([
    ('semantic_rag', "Ontology-guided"),
    ('fhir2text', "FHIR2Text"),
    ('prefiltered', "Prefiltered"),
    ('flowsheet', "Flowsheet"),
    ('ns_agent', "FHIR-Hopper (Ours)"),             # Swapped order: Full agent first
    ('embedding_csl', "FHIR-Hopper (w/o tools) (Ours)") # Ablation last
])

PRETTIFY_DATASET_NAME = {
    'ehrqa': "EHRQA",
    'fhiragentbench': "FHIRAgentBench",
    'medagentbench': "MedAgentBench"
}

ORDERED_STRATEGIES = list(PRETTIFY_STRATEGY_NAME.values())

STRATEGY_COLORS = {
    "Ontology-guided":         (235/255, 130/255, 220/255),
    "FHIR2Text":               (110/255, 215/255, 110/255),
    "Prefiltered":             (245/255, 155/255, 85/255),
    "Flowsheet":               (126/255, 97/255,  72/255),
    "FHIR-Hopper (Ours)":             (70/255, 150/255, 240/255),
    "FHIR-Hopper (w/o tools) (Ours)": (245/255,  245/255, 245/255)
}

PLOT_FIGSIZE_PER_DATASET = 7
PLOT_BAR_WIDTH = 0.7
PLOT_BAR_ALPHA = 0.8
PLOT_ERROR_CAPSIZE = 6
PLOT_ERROR_LW = 2

def prepare_plot_data(summary_df: pd.DataFrame) -> pd.DataFrame:
    df = summary_df.copy().reset_index()
    df['strategy_pretty'] = df['config_strategy_name'].map(lambda x: PRETTIFY_STRATEGY_NAME.get(x, x))
    df['dataset_pretty'] = df['config_dataset_name'].map(lambda x: PRETTIFY_DATASET_NAME.get(x, x))
    df['strategy_pretty'] = pd.Categorical(df['strategy_pretty'], categories=ORDERED_STRATEGIES, ordered=True)
    idx_best = df.groupby(['dataset_pretty', 'strategy_pretty'], observed=True)['accuracy'].idxmax()
    best_models_df = df.loc[idx_best].copy()
    best_models_df = best_models_df.sort_values(['dataset_pretty', 'strategy_pretty'])
    return best_models_df

def calculate_significance(best_models_df: pd.DataFrame, results_df: pd.DataFrame) -> dict:
    print("\n🔬 Statistical Significance Analysis:")
    dataset_sig_info = {}
    for dataset in best_models_df['dataset_pretty'].unique():
        subset = best_models_df[best_models_df['dataset_pretty'] == dataset].sort_values('accuracy', ascending=False)
        if len(subset) < 2: continue
        top1 = subset.iloc[0]
        top2_candidates = subset[(subset['strategy_pretty'] != top1['strategy_pretty']) &
                                 (subset['config_strategy_name'] != 'embedding_csl')]
        if top2_candidates.empty: continue
        top2 = top2_candidates.iloc[0]
        def get_raw_data(config):
            mask = (results_df['config_dataset_name'] == config['config_dataset_name']) & \
                   (results_df['config_strategy_name'] == config['config_strategy_name']) & \
                   (results_df['config_models_answer_model'] == config['config_models_answer_model'])
            return results_df[mask][['instance_id', 'is_correct']].copy()
        df1, df2 = get_raw_data(top1), get_raw_data(top2)
        merged = pd.merge(df1, df2, on='instance_id', suffixes=('_top1', '_top2'))
        acc1, acc2 = pd.to_numeric(merged['is_correct_top1']).fillna(0).astype(int), pd.to_numeric(merged['is_correct_top2']).fillna(0).astype(int)
        n10, n01 = sum((acc1 == 1) & (acc2 == 0)), sum((acc1 == 0) & (acc2 == 1))
        discordant_total = n10 + n01
        p_val = stats.binomtest(k=n10, n=discordant_total, p=0.5).pvalue if discordant_total > 0 else float('nan')
        star = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else ""
        print(f"Dataset: {dataset} | Best: {top1['strategy_pretty']} vs {top2['strategy_pretty']} (p={p_val:.4e})")
        dataset_sig_info[dataset] = {'top1_strategy': top1['strategy_pretty'], 'top2_strategy': top2['strategy_pretty'], 'star': star}
    return dataset_sig_info

def plot_bar_chart(best_models_df: pd.DataFrame, sig_info: dict = None):
    datasets = sorted(best_models_df['dataset_pretty'].unique())
    n_datasets = len(datasets)
    fig, axes = plt.subplots(1, n_datasets, figsize=(PLOT_FIGSIZE_PER_DATASET * n_datasets, PLOT_FIGSIZE_PER_DATASET), sharey=True)
    if n_datasets == 1: axes = [axes]
    active_strategies = [s for s in ORDERED_STRATEGIES if s in best_models_df['strategy_pretty'].unique()]
    strategy_palette = {strat: STRATEGY_COLORS[strat] for strat in active_strategies}

    for i, dataset in enumerate(datasets):
        ax = axes[i]
        subset = best_models_df[best_models_df['dataset_pretty'] == dataset]
        sns.barplot(data=subset, x='strategy_pretty', y='accuracy', hue='strategy_pretty', order=active_strategies, palette=strategy_palette, alpha=PLOT_BAR_ALPHA, dodge=False, edgecolor='black', width=PLOT_BAR_WIDTH, ax=ax)

        # Styling for ablation
        for patch in ax.patches:
            x_pos = patch.get_x() + patch.get_width()/2
            if abs(x_pos - active_strategies.index("FHIR-Hopper (w/o tools) (Ours)")) < 0.1:
                patch.set_alpha(0.3); patch.set_linestyle('--'); patch.set_linewidth(2)

        for _, row in subset.iterrows():
            ax.errorbar(x=active_strategies.index(row['strategy_pretty']), y=row['accuracy'], yerr=row['accuracy_margin'], fmt='none', c='black', capsize=PLOT_ERROR_CAPSIZE)

        if sig_info and dataset in sig_info and sig_info[dataset]['star']:
            info = sig_info[dataset]; x1, x2 = active_strategies.index(info['top1_strategy']), active_strategies.index(info['top2_strategy'])
            y = subset['accuracy'].max() + 0.1
            ax.plot([x1, x1, x2, x2], [y-0.02, y, y, y-0.02], color='black'); ax.text((x1+x2)/2, y, info['star'], ha='center', fontsize=18)

        ax.set_xlabel(dataset, fontsize=18); ax.set_xticks([]); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

    # Update y-label with requested text and font size
    axes[0].set_ylabel("Accuracy (Answer Correctness)\n", fontsize=16)
    plt.tight_layout(rect=[0, 0.08, 1, 1])
    legend_patches = [mpatches.Patch(facecolor=STRATEGY_COLORS[s], alpha=0.3 if "w/o tools" in s else 0.8, label=s, edgecolor='black', linestyle='--' if "w/o tools" in s else '-') for s in active_strategies]
    fig.legend(handles=legend_patches, loc='upper center', bbox_to_anchor=(0.5, 0.05), ncol=len(active_strategies))

    plt.savefig(
          f'all_methods_accuracy_comparisons_barplot.pdf',
          format='pdf',
          bbox_inches='tight',
      )
    plt.show(); return fig

if 'summary_df_plot' in locals() and 'results_df_common' in locals():
  fig = plot_bar_chart(prepare_plot_data(summary_df_plot), calculate_significance(prepare_plot_data(summary_df_plot), results_df_common))

In [ ]:
# @title Average Context Tokens per Strategy with Error Bars (Updated Layout)

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.patches as mpatches
import seaborn as sns
from collections import OrderedDict

# ==============================================================================
# GLOBAL CONSTANTS
# ==============================================================================
TABLE_PRETTIFY_STRATEGY = OrderedDict([
    ('semantic_rag', "Ontology-guided"),
    ('fhir2text', "FHIR2Text"),
    ('prefiltered', "Prefiltered"),
    ('flowsheet', "Flowsheet"),
    ('ns_agent', "FHIR-Hopper (Ours)"),
])

PRETTIFY_DATASET_NAME = {
    'ehrqa': "EHRQA",
    'fhiragentbench': "FHIRAgentBench",
    'medagentbench': "MedAgentBench"
}

PRETTIFY_MODEL_NAME = OrderedDict([
    ('claude-sonnet-4-6', "Claude Sonnet 4.6"),
    ('gemini-3-flash-preview', "Gemini 3 Flash"),
    ('openai/gpt-5-mini', "GPT-5 Mini")
])

ORDERED_STRATEGIES = list(TABLE_PRETTIFY_STRATEGY.values())

# Consistent strategy colors across all plots
STRATEGY_COLORS = {
    "Ontology-guided":         (235/255, 130/255, 220/255),
    "FHIR2Text":               (110/255, 215/255, 110/255),
    "Prefiltered":             (245/255, 155/255,  85/255),
    "Flowsheet":               (126/255, 97/255,  72/255),
    "FHIR-Hopper (Ours)":      (70/255, 150/255, 240/255),
}

# Textures for different models
MODEL_HATCHES = {
    "Claude Sonnet 4.6": "",   # No hatch
    "Gemini 3 Flash": "xx",   # Double cross
    "GPT-5 Mini": ".."         # Dotted
}

DATASET_TICK_INTERVALS = {
    "EHRQA": {"major": 10000, "minor": 5000},
    "FHIRAgentBench": {"major": 100000, "minor": 50000},
    "MedAgentBench": {"major": 100000, "minor": 50000}
}

# ==============================================================================
# 1. DATA PREPARATION
# ==============================================================================
def prepare_token_plot_data(results_df: pd.DataFrame) -> pd.DataFrame:
    # Using .copy() to avoid SettingWithCopyWarning
    df = results_df.copy()
    df = df[~df['config_strategy_name'].isin(['embedding_csl'])].copy()

    df['strategy_pretty'] = df['config_strategy_name'].map(lambda x: TABLE_PRETTIFY_STRATEGY.get(x, x))
    df['dataset_pretty'] = df['config_dataset_name'].map(lambda x: PRETTIFY_DATASET_NAME.get(x, x))
    df['model_pretty'] = df['config_models_answer_model'].map(lambda x: PRETTIFY_MODEL_NAME.get(x, x))

    df = df[df['strategy_pretty'].isin(ORDERED_STRATEGIES)].copy()
    df['strategy_pretty'] = pd.Categorical(df['strategy_pretty'], categories=ORDERED_STRATEGIES, ordered=True)
    df['model_pretty'] = pd.Categorical(df['model_pretty'], categories=list(PRETTIFY_MODEL_NAME.values()), ordered=True)

    df = df.sort_values(['dataset_pretty', 'strategy_pretty', 'model_pretty'])
    return df

# ==============================================================================
# 2. PLOTTING FUNCTION
# ==============================================================================
def plot_token_bar_chart(df: pd.DataFrame):
    datasets = sorted(df['dataset_pretty'].unique())
    n_datasets = len(datasets)
    fig, axes = plt.subplots(1, n_datasets, figsize=(7 * n_datasets, 7), sharey=False)
    if n_datasets == 1: axes = [axes]

    for i, dataset in enumerate(datasets):
        ax = axes[i]
        subset = df[df['dataset_pretty'] == dataset].reset_index(drop=True)

        # Map individual bar colors by strategy
        sns.barplot(
            data=subset, x='strategy_pretty', y='context_token_count', hue='model_pretty',
            alpha=0.8, dodge=True, edgecolor='black', palette='Greys',
            linewidth=1.5, errorbar=('ci', 95), capsize=0.1,
            err_kws={'linewidth': 2, 'color': 'black'}, ax=ax
        )

        # Apply colors based on strategy and hatches based on model
        if ax.containers:
            models_in_plot = list(PRETTIFY_MODEL_NAME.values())
            for container, model_name in zip(ax.containers, models_in_plot):
                hatch = MODEL_HATCHES.get(model_name, "")
                for j, patch in enumerate(container.patches):
                    if j < len(ORDERED_STRATEGIES):
                        strategy_name = ORDERED_STRATEGIES[j]
                        patch.set_facecolor(STRATEGY_COLORS.get(strategy_name, "gray"))
                    patch.set_hatch(hatch)

        if ax.get_legend() is not None: ax.get_legend().remove()

        ax.set_title(dataset, fontsize=20, pad=15)
        ax.set_xlabel("", fontsize=18)
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=12)

        if dataset in DATASET_TICK_INTERVALS:
            ax.yaxis.set_major_locator(mtick.MultipleLocator(DATASET_TICK_INTERVALS[dataset]["major"]))
            ax.yaxis.set_minor_locator(mtick.MultipleLocator(DATASET_TICK_INTERVALS[dataset]["minor"]))

        ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('{x:,.0f}'))
        ax.tick_params(axis='y', labelsize=14)
        ax.set_ylim(bottom=0)

        if i == 0: ax.set_ylabel("Token count", fontsize=16, labelpad=10)
        else: ax.set_ylabel("")

    sns.despine()
    plt.tight_layout(w_pad=4.0, rect=[0, 0.15, 1, 1])

    # Legend 1: Strategies (Colors) - use facecolor to avoid matplotlib UserWarning
    strategy_patches = [mpatches.Patch(facecolor=STRATEGY_COLORS[s], label=s, edgecolor='black') for s in ORDERED_STRATEGIES]
    # Legend 2: Models (Hatches)
    model_patches = [mpatches.Patch(facecolor='white', label=m, edgecolor='black', hatch=MODEL_HATCHES[m]) for m in PRETTIFY_MODEL_NAME.values()]

    leg1 = fig.legend(handles=strategy_patches, title="Retrieval Strategy", loc='upper center', bbox_to_anchor=(0.3, 0.08),
                      fontsize=12, title_fontsize=14, ncol=2, frameon=False)
    leg2 = fig.legend(handles=model_patches, title="Base Model (Texture)", loc='upper center', bbox_to_anchor=(0.7, 0.08),
                      fontsize=12, title_fontsize=14, ncol=2, frameon=False)

    plt.savefig('token_usage_comparison.pdf', format='pdf', bbox_inches='tight')
    plt.show(); return fig

# ==============================================================================
# 3. RUN
# ==============================================================================
if 'results_df_common' in locals():
  results_df_plot = results_df_common.copy()
  fig = plot_token_bar_chart(prepare_token_plot_data(results_df_plot))

In [ ]:
# @title Generate Pareto Plot (Context Tokens vs. Accuracy)

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.lines as mlines
import seaborn as sns
from collections import OrderedDict

# ==============================================================================
# CONFIGURATION (Synchronized with Global Naming)
# ==============================================================================
PRETTIFY_STRATEGY_NAME = OrderedDict([
    ('semantic_rag', "Ontology-guided"),
    ('fhir2text', "FHIR2Text"),
    ('prefiltered', "Prefiltered"),
    ('flowsheet', "Flowsheet"),
    ('ns_agent', "FHIR-Hopper (Ours)")
])

PRETTIFY_DATASET_NAME = {
    'ehrqa': "EHRQA",
    'fhiragentbench': "FHIRAgentBench",
    'medagentbench': "MedAgentBench"
}

PRETTIFY_MODEL_NAME = OrderedDict([
    ('claude-sonnet-4-6', "Claude Sonnet 4.6"),
    ('gemini-3-flash-preview', "Gemini 3 Flash"),
    ('openai/gpt-5-mini', "GPT-5 Mini")
])

STRATEGY_COLORS = {
    "Ontology-guided": (235/255, 130/255, 220/255),
    "FHIR2Text": (110/255, 215/255, 110/255),
    "Prefiltered": (245/255, 155/255,  85/255),
    "Flowsheet": (126/255, 97/255,  72/255),
    "FHIR-Hopper (Ours)": (70/255, 150/255, 240/255)
}

# Assign distinct shapes to the models
MODEL_MARKERS = {
    "Claude Sonnet 4.6": "s",    # Square
    "Gemini 3 Flash": "o",       # Circle
    "GPT-5 Mini": "^"            # Triangle
}

# ==============================================================================
# 1. DATA PREPARATION & PARETO LOGIC
# ==============================================================================
def prepare_pareto_data(summary_df: pd.DataFrame) -> pd.DataFrame:
    df = summary_df.copy().reset_index()

    # Filter and map names to match other plots
    df = df[df['config_strategy_name'].isin(PRETTIFY_STRATEGY_NAME.keys())]
    df['Strategy'] = df['config_strategy_name'].map(PRETTIFY_STRATEGY_NAME)
    df['Dataset'] = df['config_dataset_name'].map(PRETTIFY_DATASET_NAME)
    df['Base Model'] = df['config_models_answer_model'].map(PRETTIFY_MODEL_NAME)

    return df

def get_pareto_frontier(Xs, Ys):
    """
    Finds the Pareto frontier where we want to MINIMIZE X (Tokens) and MAXIMIZE Y (Accuracy).
    """
    points = sorted(zip(Xs, Ys), key=lambda p: (p[0], -p[1]))

    pareto_X, pareto_Y = [], []
    max_y = -np.inf

    for x, y in points:
        if y > max_y:
            pareto_X.append(x)
            pareto_Y.append(y)
            max_y = y

    return pareto_X, pareto_Y

# ==============================================================================
# 2. PLOTTING FUNCTION
# ==============================================================================
def plot_pareto_frontier(df: pd.DataFrame):
    datasets = sorted(df['Dataset'].dropna().unique())
    n_datasets = len(datasets)

    fig, axes = plt.subplots(1, n_datasets, figsize=(7 * n_datasets, 6), sharey=True)
    if n_datasets == 1: axes = [axes]

    for i, dataset in enumerate(datasets):
        ax = axes[i]
        subset = df[df['Dataset'] == dataset].dropna(subset=['mean_context_token_count', 'accuracy'])

        # Draw Pareto Line first
        pX, pY = get_pareto_frontier(subset['mean_context_token_count'].values, subset['accuracy'].values)
        ax.plot(pX, pY, linestyle='--', color='gray', linewidth=2, alpha=0.7, zorder=1)

        # Scatter Plot
        sns.scatterplot(
            data=subset,
            x='mean_context_token_count',
            y='accuracy',
            hue='Strategy',
            style='Base Model',
            palette=STRATEGY_COLORS,
            markers=MODEL_MARKERS,
            s=200,
            edgecolor='black',
            linewidth=1.2,
            alpha=0.9,
            ax=ax,
            zorder=2
        )

        # Formatting Axes
        ax.set_title(dataset, fontsize=18, pad=15)
        ax.set_xlabel("Mean Context Tokens", fontsize=16, labelpad=10)
        ax.spines['bottom'].set_linewidth(1.5)
        ax.spines['left'].set_linewidth(1.5)

        ax.xaxis.set_major_formatter(mtick.StrMethodFormatter('{x:,.0f}'))
        ax.tick_params(axis='both', which='major', labelsize=13)
        ax.grid(True, linestyle='--', alpha=0.5, zorder=0)

        if i == 0:
            ax.set_ylabel("Accuracy", fontsize=16, labelpad=10)
            ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0, symbol='%'))
        else:
            ax.set_ylabel("")

        if ax.get_legend() is not None:
            ax.get_legend().remove()

    sns.despine()
    plt.tight_layout(w_pad=3.0, rect=[0, 0.15, 1, 1])

    # Global Legend Construction
    strategy_handles = [
        mlines.Line2D([], [], color=color, marker='o', linestyle='None',
                      markersize=12, markeredgecolor='black', label=strat)
        for strat, color in STRATEGY_COLORS.items() if strat in df['Strategy'].unique()
    ]

    model_handles = [
        mlines.Line2D([], [], color='gray', marker=shape, linestyle='None',
                      markersize=12, markeredgecolor='black', label=model)
        for model, shape in MODEL_MARKERS.items() if model in df['Base Model'].unique()
    ]

    pareto_handle = [mlines.Line2D([], [], color='gray', linestyle='--', linewidth=2, label="Pareto Frontier")]

    fig.legend(handles=strategy_handles, title="Retrieval Strategy",
               loc='upper center', bbox_to_anchor=(0.3, 0.08),
               fontsize=13, title_fontsize=14, ncol=2)

    fig.legend(handles=model_handles + pareto_handle, title="Base Model & Frontier",
               loc='upper center', bbox_to_anchor=(0.7, 0.08),
               fontsize=13, title_fontsize=14, ncol=2)

    plt.show()
    return fig

# ==============================================================================
# RUN
# ==============================================================================
if 'summary_df_plot' in locals():
  df_pareto = prepare_pareto_data(summary_df_plot)
  fig_pareto = plot_pareto_frontier(df_pareto)

  # Export Figure
  PLOT_DIR = "outputs/paper_plots/"
  os.makedirs(PLOT_DIR, exist_ok=True)
  save_path = os.path.join(PLOT_DIR, 'pareto_token_vs_accuracy.pdf')
  fig_pareto.savefig(save_path, format='pdf', bbox_inches='tight')
  print(f"✅ Pareto plot saved to: {save_path}")

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.lines as mlines
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from collections import OrderedDict

# ==============================================================================
# CONFIGURATION
# ==============================================================================
PRETTIFY_STRATEGY_NAME = OrderedDict([
    ('semantic_rag', "Ontology-guided"),
    ('fhir2text', "FHIR2Text"),
    ('prefiltered', "Prefiltered"),
    ('flowsheet', "Flowsheet"),
    ('ns_agent', "FHIR-Hopper (Ours)")
])

PRETTIFY_DATASET_NAME = {
    'ehrqa': "EHRQA",
    'fhiragentbench': "FHIRAgentBench",
    'medagentbench': "MedAgentBench"
}

PRETTIFY_MODEL_NAME = {
    'claude-sonnet-4-6': "Claude Sonnet 4.6",
    'openai/gpt-5-mini': "GPT-5 Mini",
    'gemini-3-flash-preview': "Gemini 3 Flash",
}

# Updated Markers to avoid confusion with strategy 'circles'
MODEL_MARKERS = {
    "Claude Sonnet 4.6": "D",       # Diamond
    "Gemini 3 Flash": "H",          # Hexagon
    "GPT-5 Mini": "s",              # Square
    "Default": "*"
}

STRATEGY_COLORS = {
    "Ontology-guided": (235/255, 130/255, 220/255),
    "FHIR2Text": (110/255, 215/255, 110/255),
    "Prefiltered": (245/255, 155/255,  85/255),
    "Flowsheet": (126/255, 97/255,  72/255),
    "FHIR-Hopper (Ours)": (70/255, 150/255, 240/255)
}

# ==============================================================================
# 1. DATA PREPARATION
# ==============================================================================
def prepare_efficiency_data(summary_df: pd.DataFrame) -> pd.DataFrame:
    df = summary_df.copy().reset_index()
    df = df[df['config_strategy_name'].isin(PRETTIFY_STRATEGY_NAME.keys())]
    df['Strategy'] = df['config_strategy_name'].map(PRETTIFY_STRATEGY_NAME)
    df['Dataset'] = df['config_dataset_name'].map(PRETTIFY_DATASET_NAME)

    if 'config_models_answer_model' in df.columns:
        df['Model'] = df['config_models_answer_model'].map(PRETTIFY_MODEL_NAME)
    else:
        df['Model'] = "Default"

    df['token_efficiency'] = 1_000_000 / df['mean_context_token_count']
    df['Strategy'] = pd.Categorical(df['Strategy'], categories=list(PRETTIFY_STRATEGY_NAME.values()), ordered=True)
    return df

# ==============================================================================
# 2. PLOTTING FUNCTION
# ==============================================================================
def get_pareto_front(x, y):
    points = sorted(zip(x, y), key=lambda p: p[0])
    if not points: return [], []
    pareto_x, pareto_y = [], []
    max_y = -np.inf
    for curr_x, curr_y in reversed(points):
        if curr_y > max_y:
            pareto_x.append(curr_x)
            pareto_y.append(curr_y)
            max_y = curr_y
    return zip(*sorted(zip(pareto_x, pareto_y)))

def plot_efficiency_scatter(df: pd.DataFrame, show_pareto=True):
    datasets = sorted(df['Dataset'].dropna().unique())
    n_datasets = len(datasets)
    fig, axes = plt.subplots(1, n_datasets, figsize=(7 * n_datasets, 6), sharey=False)
    if n_datasets == 1: axes = [axes]

    active_markers = {mod: MODEL_MARKERS.get(mod, MODEL_MARKERS["Default"]) for mod in df['Model'].unique()}
    bg_cmap = LinearSegmentedColormap.from_list('perf_grad', ['#fff5f5', '#ffffff', '#f0fff0'])

    for i, dataset in enumerate(datasets):
        ax = axes[i]
        subset = df[df['Dataset'] == dataset].dropna(subset=['token_efficiency', 'accuracy'])

        if show_pareto:
            px, py = get_pareto_front(subset['token_efficiency'], subset['accuracy'])
            if px:
                ax.plot(px, py, color='black', linestyle='-', linewidth=2.5, alpha=0.3, zorder=1, drawstyle='steps-post')

        sns.scatterplot(
            data=subset, x='token_efficiency', y='accuracy', hue='Strategy', style='Model',
            palette=STRATEGY_COLORS, markers=active_markers, s=400, edgecolor='black',
            linewidth=1.5, alpha=0.8, ax=ax, zorder=4
        )

        xlim, ylim = ax.get_xlim(), ax.get_ylim()
        # Add small vertical padding to prevent clipping at the top
        ax.set_ylim(ylim[0], ylim[1] + 0.05)
        new_ylim = ax.get_ylim()

        nx, ny = 100, 100
        X, Y = np.meshgrid(np.linspace(0, 1, nx), np.linspace(0, 1, ny))
        ax.imshow(X + Y, extent=[xlim[0], xlim[1], new_ylim[0], new_ylim[1]], origin='lower', cmap=bg_cmap, aspect='auto', zorder=-1)
        ax.set_xlim(xlim); ax.set_ylim(new_ylim)

        if i == 0: ax.annotate("", xy=(0.96, 0.96), xycoords='axes fraction', xytext=(0.88, 0.86), arrowprops=dict(facecolor='black', alpha=0.6, width=3, headwidth=10))

        ax.set_title(dataset, fontsize=18, pad=15)
        ax.set_xlabel("Token Efficiency\n(Instances Processed per 1M Tokens)", fontsize=16, labelpad=10)
        ax.xaxis.set_major_formatter(mtick.StrMethodFormatter('{x:,.0f}'))
        ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0, symbol='%'))
        ax.grid(True, linestyle='--', alpha=0.3, zorder=0)

        if i == 0:
          ax.set_ylabel("Accuracy", fontsize=16, labelpad=10)
        else:
          ax.set_ylabel("")

        if ax.get_legend() is not None: ax.get_legend().remove()

    sns.despine()
    plt.tight_layout(w_pad=3.0, rect=[0, 0.16, 1, 1])

    # Legend 1: Strategies (Colors)
    strategy_handles = [mlines.Line2D([], [], color=color, marker='o', linestyle='None', markersize=12, markeredgecolor='black', label=strat)
                        for strat, color in STRATEGY_COLORS.items() if strat in df['Strategy'].unique()]

    # Legend 2: Models & Pareto
    model_handles = [mlines.Line2D([], [], color='gray', marker=marker, linestyle='None', markersize=12, markeredgecolor='black', label=mod)
                     for mod, marker in active_markers.items()]
    pareto_handle = [mlines.Line2D([], [], color='black', alpha=0.3, linewidth=2.5, label='Pareto Front')] if show_pareto else []

    leg1 = fig.legend(handles=strategy_handles, title="Retrieval Strategy", loc='upper center', bbox_to_anchor=(0.35, 0.12),
                      fontsize=13, title_fontsize=14, ncol=2, frameon=False)

    leg2 = fig.legend(handles=model_handles + pareto_handle, title="Base Model & Frontier", loc='upper center', bbox_to_anchor=(0.7, 0.12),
                      fontsize=13, title_fontsize=14, ncol=2, frameon=False)

    plt.savefig('all_pareto_frontier_scatter.pdf', format='pdf', bbox_inches='tight')
    plt.show(); return fig

# ==============================================================================
# RUN THE PIPELINE
# ==============================================================================
if 'results_df_common' in locals() and 'calculate_summay_stats' in locals():
  results_df_plot = results_df_common.copy()
  results_df_plot = results_df_plot[results_df_plot['config_strategy_name'] != 'embedding_csl']
  summary_df_plot = results_df_plot.groupby(["config_dataset_name", "config_strategy_name", "config_models_answer_model"]).apply(calculate_summay_stats, include_groups=False)
  df_efficiency = prepare_efficiency_data(summary_df_plot)
  fig_efficiency = plot_efficiency_scatter(df_efficiency, show_pareto=True)

# Generate Token Count Distributions for All Datasets

In [ ]:
import json
import os
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pandas as pd
import seaborn as sns
import tqdm

from fhir_retrieval_bench.utils import api

# --- Setup API Client ---
genai_keys = [k.strip() for k in os.getenv("GENAI_API_KEYS", "").split(",") if k.strip()]
creds = api.Credentials.from_lists(genai_api_keys=genai_keys)
MODEL_NAME = "gemini-3-flash-preview"  # Model used for token counting
BACKEND = "public"

api_client = api.get_api_client(MODEL_NAME, creds, backend=BACKEND) if genai_keys else None

# --- Data Paths ---
PATHS = {
    "FHIRAgentBench": "data/fhiragentbench/fhir_bundles.pq",
    "FHIRPathQA": "data/fhirpathqa/fhir_bundles.pq",
    "MedAgentBench": "data/medagentbench/fhir_bundles.pq",
}

# --- Process Datasets ---
bundle_data = {}

for dataset_name, path in PATHS.items():
    if not os.path.exists(path):
        print(f"WARNING: Path {path} not found.")
        continue
    print(f"Loading: {path}")
    with open(path, "rb") as infile:
        df = pd.read_parquet(infile)
    bundle_data[dataset_name] = df

In [ ]:
def get_token_counts_api(df, client, model_name, sample_size=None):
    """Computes token counts using the API for a provided DataFrame."""
    token_counts = []

    # Optional: Take a sample to avoid hitting rate limits or long wait times
    if sample_size and sample_size < len(df):
        print(f"Sampling {sample_size} records out of {len(df)}")
        df = df.sample(n=sample_size, random_state=42)

    if not client:
        print("API Client not available, skipping token counting.")
        return []

    for _, row in tqdm.tqdm(df.iterrows(), total=len(df), desc="Counting Tokens"):
        bundle_content = row["FHIR Bundle"]

        if isinstance(bundle_content, dict):
            bundle_str = json.dumps(bundle_content)
        else:
            bundle_str = str(bundle_content)

        try:
            # Call the repository's API wrapper
            num_tokens = client.count_tokens(model_name, bundle_str)
            if num_tokens is not None:
                token_counts.append(num_tokens)
            else:
                print("API returned None for token count")
        except Exception as e:
            print(f"Error during API call: {e}")

    return token_counts

plot_data = []

# Set sample_size to None to process all, or an integer to test quickly
SAMPLE_SIZE = None

for dataset_name, bundle_df in bundle_data.items():
    print(f"Loading externally: {path}")

    # Pass the loaded DataFrame into the token counting function
    counts = get_token_counts_api(
        bundle_df, api_client, MODEL_NAME, sample_size=SAMPLE_SIZE
    )

    for count in counts:
        plot_data.append({"Dataset": dataset_name, "Token Count": count})

# Now `plot_data` is ready for sns/plt usage

In [ ]:
df_plot = pd.DataFrame(plot_data)

df_plot = df_plot[
    (df_plot["Dataset"]!="MedAgentBench") |
    ((df_plot["Dataset"]=="MedAgentBench") & (df_plot["Token Count"]>5000))
]

plt.figure()
sns.set_theme(style="whitegrid", font="Google Sans")

ax = sns.boxplot(
    x="Dataset",
    y="Token Count",
    data=df_plot,
    palette="Set2",
    width=0.6,
    showmeans=True,
    meanprops={
        "marker": "o",
        "markerfacecolor": "white",
        "markeredgecolor": "black",
        "markersize": "8",
    },
)

plt.xlabel("Dataset", fontsize=14, labelpad=15)
plt.ylabel("Token Count", fontsize=14,  labelpad=15)

ax.yaxis.set_major_formatter(ticker.StrMethodFormatter("{x:,.0f}"))
plt.tight_layout()
plt.show()

In [ ]:
import tiktoken
import tqdm

# Use tiktoken as standard public tokenizer replacement for token counts
try:
  spt = tiktoken.get_encoding("cl100k_base")
except Exception:
  spt = None

In [ ]:
# @title Plotting Utility Function
import matplotlib.ticker as ticker

def plot_token_distribution(token_counts, font_family="Google Sans",
                            title_plot_str=None,
                            dataset_string=None):
  if font_family:
    sns.set_theme(style="white", font=font_family)
  else:
    sns.set_theme(style="white")

  # Create the figure
  plt.figure(figsize=(14, 7), dpi=500)

  # Plot the distribution: Histogram + KDE
  # We use a nice blue color and add some transparency
  # Added bins=30 for more granularity
  ax = sns.histplot(token_counts,
                    kde=True, bins=50, color="skyblue", alpha=0.6, line_kws={'linewidth': 2})

  sns.rugplot(token_counts, color="darkblue", height=0.05, alpha=0.5)

  mean_val = np.mean(token_counts)
  median_val = np.median(token_counts)

  plt.axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_val:,.0f}')
  plt.axvline(median_val, color='green', linestyle='-', linewidth=2, label=f'Median: {median_val:,.0f}')

  if title_plot_str:
    plt.title(title_plot_str, fontsize=18, fontweight='bold', pad=20)
  else:
    plt.title(f'Distribution of Token Counts per Patient Bundle (n={len(token_counts)})\n', fontsize=18, fontweight='bold', pad=20)

  plt.xlabel('\nToken Count', fontsize=14)
  plt.ylabel('Frequency\n', fontsize=14)
  plt.legend(fontsize=12)

  plt.xlim(left=10000)

  plt.gca().xaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))
  plt.savefig(
          f'{dataset_string}_token_distribution_count.pdf',
          format='pdf',
          bbox_inches='tight',
      )

  plt.tight_layout()
  plt.show()

## EHRQA

In [ ]:
PATH_TO_READ = "data/ehrqa/dictionary_of_synthea_patients_for_ehrqa_validation.pkl"
if not os.path.exists(PATH_TO_READ):
  print(f"WARNING: {PATH_TO_READ} not found.")
else:
  with open(PATH_TO_READ, "rb") as infile:
    patient_id_to_json = pickle.load(infile)

In [ ]:
ehrqa_fhir_conunts = []
if spt is not None and 'patient_id_to_json' in locals():
  for patient_id in tqdm.tqdm(patient_id_to_json.keys()):
    patient_json_str = str(patient_id_to_json[patient_id])
    tokens = spt.encode(patient_json_str)
    ehrqa_fhir_conunts.append(len(tokens))

In [ ]:
plot_token_distribution(ehrqa_fhir_conunts,
                        title_plot_str="Distribution of FHIR Bundle Token Counts Across 20 Patient Profiles in EHRQA",
                        dataset_string="ehrqa")

## MedAgentBench

In [ ]:
PATH_TO_MEDAGENT_DATA = "data/medagentbench/fhir_bundles.pq"
if not os.path.exists(PATH_TO_MEDAGENT_DATA):
  print(f"WARNING: {PATH_TO_MEDAGENT_DATA} not found.")
else:
  with open(PATH_TO_MEDAGENT_DATA, "rb") as infile:
    medagent_fhir_df = pd.read_parquet(infile)

  medagent_fhir_token_conunts = []
  if spt is not None:
    for row, col in tqdm.tqdm(medagent_fhir_df.iterrows(), total=len(medagent_fhir_df)):
      current_fhir_bundle = str(col['FHIR Bundle'])
      number_of_tokens = len(spt.encode(current_fhir_bundle))
      if number_of_tokens > 5000:
        medagent_fhir_token_conunts.append(number_of_tokens)

In [ ]:
plot_token_distribution(medagent_fhir_token_conunts,
                        title_plot_str="Distribution of FHIR Bundle Token Counts Across 100 Patient Profiles in MedAgentBench",
                        dataset_string="medagentbench")

## FHIR AgentBench

In [ ]:
PATH_TO_MIMIC_DATA = "data/fhiragentbench/fhir_bundles.pq"
if not os.path.exists(PATH_TO_MIMIC_DATA):
  print(f"WARNING: {PATH_TO_MIMIC_DATA} not found.")
else:
  with open(PATH_TO_MIMIC_DATA, "rb") as infile:
    mimic_fhir_df = pd.read_parquet(infile)

  mimic_fhir_token_conunts = []
  if spt is not None:
    for row, col in tqdm.tqdm(mimic_fhir_df.iterrows(), total=len(mimic_fhir_df)):
      current_fhir_bundle = str(col['FHIR Bundle'])
      number_of_tokens = len(spt.encode(current_fhir_bundle))
      mimic_fhir_token_conunts.append(number_of_tokens)

In [ ]:
plot_token_distribution(mimic_fhir_token_conunts,
                        title_plot_str="Distribution of FHIR Bundle Token Counts Across 94 Patients in FHIR AgentBench",
                        dataset_string="fhiragentbench")